# **4. Финальное обучение**

**Цель:** обучить модели на новых сгенерированных признаках, сравнить с бейзлайном и сделать выводы.

In [2]:
import sys
import os

# необходимо для того чтобы ноутбук увидел src
current_dir = os.getcwd()

if current_dir.endswith('notebooks'):
    project_root = os.path.dirname(current_dir)
else:
    project_root = current_dir

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
import matplotlib.pyplot as plt
import warnings

from src.data_loader import load_train, load_events
from src.features_final import build_features
from src.metrics import precision_at_recall

warnings.filterwarnings('ignore')

N_SPLITS = 5
RANDOM_STATE = 42

Загрузка данных:

In [4]:
train, events = load_train(), load_events()
features = build_features(events, train, verbose=True)
features.head()

Событий в окне: 198504
Итого признаков: 61


,cookie_id,event_count,unique_items,unique_categories,unique_locations,unique_event_types,unique_search_queries_x,items_per_event,categories_per_event,locations_per_event,...,item_view_to_photo_ratio,bigrams_per_transition,unique_trigrams,trigrams_per_transition,ptr_std_x,ptr_std_y,ptr_range_x,ptr_range_y,ptr_mean_step,target
0,ck_000c95f1408dcb00,16,9,2,8,4,3,0.562500,0.125000,0.500000,...,5.500000,0.466667,8,0.571429,479.263171,327.339363,1882.0,953.0,646.201297,0
1,ck_000e8c52636e3bec,34,20,4,12,7,8,0.588235,0.117647,0.352941,...,1.833333,0.606061,31,0.968750,552.412738,273.203362,1892.0,1020.0,786.870245,0
2,ck_0010e31baa4a1fb7,16,8,3,4,6,4,0.500000,0.187500,0.250000,...,3.500000,0.733333,14,1.000000,485.953814,349.957118,1450.0,1021.0,772.326478,0
3,ck_0010ec3874fb5378,74,31,2,19,6,8,0.418919,0.027027,0.256757,...,1.666667,0.287671,46,0.638889,NaN,NaN,NaN,NaN,NaN,0
4,ck_001722063b94cae0,15,8,3,9,4,4,0.533333,0.200000,0.600000,...,4.000000,0.571429,10,0.769231,NaN,NaN,NaN,NaN,NaN,0


In [5]:
X = features.drop(columns=['cookie_id', 'target'])
y = features['target']

Общие функции для валидации:

In [6]:
def evaluate_cv(model, X: pd.DataFrame, y: pd.Series, n_splits: int = N_SPLITS):
    """OOF-оценка модели. Метрика считается один раз на всех OOF-score"""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    oof = np.zeros(len(X))
    fold_iters = list(skf.split(X, y))

    for fold, (tr_idx, va_idx) in enumerate(fold_iters, 1):
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
        print(f"  fold {fold}/{n_splits} - обучено")

    y_arr = y.values
    p_at_r = precision_at_recall(y_arr, oof)
    pr_auc = average_precision_score(y_arr, oof)
    roc_auc = roc_auc_score(y_arr, oof)

    return {
        "P@R>=0.7": p_at_r,
        "PR-AUC": pr_auc,
        "ROC-AUC": roc_auc,
        "oof": oof,
    }


def constant_baseline(y: pd.Series) -> float:
    """Нижняя граница: все куки получают одинаковый score."""
    score = np.full(len(y), 1.0)
    return precision_at_recall(y.values, score)

### **4.1. Обучение логистической регрессии**

In [6]:
logreg = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=3000,
            penalty='elasticnet',
            solver='saga',
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])

# Сравнение с константным baseline
const_score = constant_baseline(y)
print(f"\nConstant baseline P@R>=0.7: {const_score:.4f}")

# Кросс-валидация
print(f"\nStratifiedKFold({N_SPLITS}) - LogisticRegression:")
lr_res = evaluate_cv(logreg, X, y, n_splits=N_SPLITS)

print("\nРезультат:\n")
print(f"  Precision@Recall>=0.70 : {lr_res['P@R>=0.7']:.4f}")
print(f"  PR-AUC                 : {lr_res['PR-AUC']:.4f}")
print(f"  ROC-AUC                : {lr_res['ROC-AUC']:.4f}")


Constant baseline P@R>=0.7: 0.0811

StratifiedKFold(5) - LogisticRegression:
  fold 1/5 - обучено
  fold 2/5 - обучено
  fold 3/5 - обучено
  fold 4/5 - обучено
  fold 5/5 - обучено

Результат:

  Precision@Recall>=0.70 : 0.5307
  PR-AUC                 : 0.6574
  ROC-AUC                : 0.9039


### **4.2. Обучение случайного леса**

In [7]:
random_forest = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))
    ])

const_score = constant_baseline(y)
print(f"\nConstant baseline P@R>=0.7: {const_score:.4f}")

print(f"\nStratifiedKFold({N_SPLITS}) - RandomForestClassifier:\n")
rf_res = evaluate_cv(random_forest, X, y, n_splits=N_SPLITS)

print("\nРезультат:\n")
print(f"  Precision@Recall>=0.70 : {rf_res['P@R>=0.7']:.4f}")
print(f"  PR-AUC                 : {rf_res['PR-AUC']:.4f}")
print(f"  ROC-AUC                : {rf_res['ROC-AUC']:.4f}")      


Constant baseline P@R>=0.7: 0.0811

StratifiedKFold(5) - RandomForestClassifier:

  fold 1/5 - обучено
  fold 2/5 - обучено
  fold 3/5 - обучено
  fold 4/5 - обучено
  fold 5/5 - обучено

Результат:

  Precision@Recall>=0.70 : 0.7055
  PR-AUC                 : 0.7531
  ROC-AUC                : 0.9244


### **4.3. Обучание градиентного бустинга**

In [8]:
cat_boost = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", CatBoostClassifier(
            iterations=700,
            depth=6,
            learning_rate=0.05,
            l2_leaf_reg=3.0,
            loss_function="Logloss",
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        )),
    ])

const_score = constant_baseline(y)
print(f"\nConstant baseline P@R>=0.7: {const_score:.4f}")

print(f"\nStratifiedKFold({N_SPLITS}) - CatBoostClassifier:\n")
ct_res = evaluate_cv(cat_boost, X, y, n_splits=N_SPLITS)

print("\nРезультат:\n")
print(f"  Precision@Recall>=0.70 : {ct_res['P@R>=0.7']:.4f}")
print(f"  PR-AUC                 : {ct_res['PR-AUC']:.4f}")
print(f"  ROC-AUC                : {ct_res['ROC-AUC']:.4f}")  


Constant baseline P@R>=0.7: 0.0811

StratifiedKFold(5) - CatBoostClassifier:

  fold 1/5 - обучено
  fold 2/5 - обучено
  fold 3/5 - обучено
  fold 4/5 - обучено
  fold 5/5 - обучено

Результат:

  Precision@Recall>=0.70 : 0.7503
  PR-AUC                 : 0.7744
  ROC-AUC                : 0.9335


In [9]:
results = [
    {"Model": "LogReg",  **{k: lr_res[k] for k in ("P@R>=0.7", "PR-AUC", "ROC-AUC")}},
    {"Model": "RandomForest",  **{k: rf_res[k] for k in ("P@R>=0.7", "PR-AUC", "ROC-AUC")}},
    {"Model": "CatBoost",      **{k: ct_res[k] for k in ("P@R>=0.7", "PR-AUC", "ROC-AUC")}},
]

df = pd.DataFrame(results).rename(columns={
    "P@R>=0.7": "P@R>=0.70",
    "PR-AUC":   "PR-AUC",
    "ROC-AUC":  "ROC-AUC",
})
df = df.sort_values("P@R>=0.70", ascending=False).reset_index(drop=True)

df.loc[len(df)] = ["Constant baseline", const_score, np.nan, np.nan]

print("=" * 60)
print(f"{'Model':<20} {'P@R>=0.70':>10} {'PR-AUC':>10} {'ROC-AUC':>10}")
print("-" * 60)
for _, row in df.iterrows():
    pr  = "-" if pd.isna(row["PR-AUC"])  else f"{row['PR-AUC']:.4f}"
    roc = "-" if pd.isna(row["ROC-AUC"]) else f"{row['ROC-AUC']:.4f}"
    print(f"{row['Model']:<20} {row['P@R>=0.70']:>10.4f} {pr:>10} {roc:>10}")
print("=" * 60)

Model                 P@R>=0.70     PR-AUC    ROC-AUC
------------------------------------------------------------
CatBoost                 0.7503     0.7744     0.9335
RandomForest             0.7055     0.7531     0.9244
LogReg                   0.5307     0.6574     0.9039
Constant baseline        0.0811          -          -


In [10]:
final_cb = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", CatBoostClassifier(
            iterations=700,
            depth=4,
            learning_rate=0.07,
            l2_leaf_reg=3.0,
            loss_function="Logloss",
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        )),
    ])

# Обучаем CatBoost на всём train - только для importance
final_cb.fit(X, y)

cat_clf = final_cb.named_steps["clf"]
importance = pd.Series(
    cat_clf.get_feature_importance(),
    index=X.columns,
).sort_values(ascending=False)

TOP_N = 20
top = importance.head(TOP_N)

print(f"Топ-{TOP_N} признаков (CatBoost feature importance):")
for name, val in top.items():
    print(f"  {name:<38} {val:>6.2f}")

print(f"\nВсего признаков: {len(importance)}")
print(f"Вносят >1%:       {(importance > 1.0).sum()}")
print(f"Вносят >0.5%:     {(importance > 0.5).sum()}")
print(f"Вносят <0.1%:     {(importance < 0.1).sum()}")

Топ-20 признаков (CatBoost feature importance):
  median_interval                          5.93
  ptr_mean_step                            4.81
  ratio_gt_60s                             4.45
  ptr_std_x                                4.36
  ptr_range_x                              4.11
  unique_categories                        3.73
  iqr_over_median                          3.62
  items_per_event                          3.61
  unique_locations                         3.41
  iqr_interval                             3.06
  p10_interval                             2.95
  cookie_age_log                           2.87
  ratio_photo_swipe                        2.85
  mean_search_page                         2.61
  categories_per_event                     2.59
  std_over_median                          2.42
  p90_interval                             2.40
  p90_over_p10                             2.25
  locations_per_event                      2.19
  ptr_range_y                           

**Результаты:**

1. Сравнение baseline и финальной версии:

    - Прирост **+0.18 абсолютных (+33% относительно)** от добавления новых фич.

2. Что дало прирост

    - **Pointer-фичи** - главный вклад. `ptr_std_x` (6.5%), `ptr_mean_step` (3.4%), `ptr_range_x` (3.4%) в топ-3 важности. У ботов курсор двигается в узкой области и короткими шагами.

    - **Интервальные фичи** - `ratio_gt_60s`, `iqr_over_median`, `p10/p90_interval` в топ-20. Боты работают с более равномерными и короткими паузами.

    - **Нормализованные счётчики** - `items_per_event`, `categories_per_event` работают лучше сырых `nunique`.

3. Что не сработало / под вопросом:

    - **Transition-фичи** (`bigrams_per_transition`, `trigrams_per_transition`, `unique_trigrams`) не попали в топ-20. Следует доработать признаки.

    - **UA-фичи** (`ratio_is_bot_ua`, `ua_length_median`) так и не удалось вытащить оттуда ничего полезного, стоит провести более тщательный анализ.

**Итог**: удалось достаточно неплохое рабочее решение, однако данная задача достаточно творческая в плане feature engineering, поэтому дальнейшие исследования также способны поднять качество. 